# Compute-Normalized Blindspot Discovery (Colab)


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/waymax-simulation-experiments/blob/main/experiments/closedloop-evaluation/notebooks/compute_normalized_blindspot_discovery_colab.ipynb)


## Research Questions
1. Which method discovers more unique blindspots per compute budget?
2. Is ranking stable across risk/blindspot definition changes?
3. Do gains remain under realism/plausibility constraints?

### Repo-Inspired Components
- **STRIVE**: challenge-plus-plausibility framing and cluster/scene diversity analysis.
- **FREA**: feasibility-guided adversariality (plausibility filtering with feasibility constraints).
- **SEAL/CAT metrics**: realism gap checks via distribution distances.
- **VerifAI/Scenic**: multi-objective rulebook ranking and coverage-style analysis.


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/achyutmorang/waymax-simulation-experiments.git"
REPO_DIR = "/content/waymax-simulation-experiments"

if os.path.exists(REPO_DIR):
    print("Repo exists, fast-forwarding main...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())


In [ ]:
# Mount Google Drive in Colab
import os

if os.environ.get('COLAB_RELEASE_TAG'):
    try:
        from google.colab import drive
        if not os.path.ismount('/content/drive'):
            drive.mount('/content/drive')
        else:
            print('[drive] /content/drive already mounted')
    except Exception as e:
        print('[drive] mount skipped:', e)
else:
    print('[drive] non-Colab environment; skipping mount')


In [ ]:
import json
import sys
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.eval_compute_normalized_discovery import (
    DiscoveryHypothesisConfig,
    NaturalnessConfig,
    best_method_per_definition,
    cluster_coverage_diversity,
    combine_repo_inspired_method_table,
    discover_and_load_run,
    evaluate_discovery_grid,
    evaluate_discovery_hypotheses,
    method_score_table,
    plausibility_filtered_summary,
    plot_definition_heatmap,
    plot_method_score_distribution,
    plot_rank_heatmap,
    plot_time_to_k,
    realism_gap_summary,
    rulebook_lexicographic_ranking,
    save_figure,
    standard_blindspot_definitions,
    standard_risk_definitions,
)

pd.set_option('display.max_columns', 300)


In [ ]:
# ===== User config =====
RUN_TAG = "closedloop_v1"
SIM_PERSIST_ROOT = "/content/drive/MyDrive/waymax_experiments/closedloop_runs/v1"
N_SHARDS = 5
OUT_RUN_TAG = None
PREFER_KIND = "merged"
REQUIRE_TRACE = True

COMPUTE_COL = None
K_VALUES = (10, 25)

# Definition robustness sweeps
RISK_SEVERITY_QUANTILES = [0.85, 0.90, 0.95]
NEAR_MISS_THRESHOLDS = [1.5, 2.0, 2.5]
BLINDSPOT_RARE_QUANTILES = [0.10, 0.20, 0.30]
STABILITY_MIN_RATES = [0.50, 0.60, 0.70]

# FREA-inspired plausibility constraints
NATURALNESS_CONFIG = NaturalnessConfig(
    max_delta_l2_abs=None,
    max_delta_l2_quantile=0.95,
    max_fallback_ratio=0.80,
    max_actor_fallback_ratio=0.80,
    require_rollout_feasible=True,
    require_no_feasibility_violation=True,
    min_dist_non_null_ratio=0.05,
)

# VerifAI-inspired multi-objective rulebook priorities
RULEBOOK_PRIORITIES = [
    "discovery_efficiency",           # maximize
    "plausible_efficiency",           # maximize
    "coverage_ratio",                 # maximize
    "wd_mean",                        # minimize
]
RULEBOOK_MAXIMIZE = {
    "discovery_efficiency": True,
    "plausible_efficiency": True,
    "coverage_ratio": True,
    "wd_mean": False,
}

HYPOTHESIS_CONFIG = DiscoveryHypothesisConfig(
    treatment="joint",
    control="random",
    score_col="discovery_efficiency",
    alpha=0.05,
    min_win_fraction=0.60,
    min_top_win_fraction=0.50,
    max_rank_std=1.00,
    min_worst_case_delta=0.0,
)

EXPORT_DIR = f"/content/drive/MyDrive/waymax_experiments/compute_normalized_blindspot_discovery/{RUN_TAG}"


In [ ]:
run_data = None
try:
    run_data = discover_and_load_run(
        run_tag=RUN_TAG,
        persist_root=SIM_PERSIST_ROOT,
        n_shards=N_SHARDS,
        out_run_tag=OUT_RUN_TAG,
        prefer_kind=PREFER_KIND,
        require_trace=REQUIRE_TRACE,
    )
except FileNotFoundError as e:
    print("[precheck] No completed simulation artifacts were found for this RUN_TAG/SIM_PERSIST_ROOT.")
    print("[precheck] Run closedloop_simulation_colab.ipynb first, or update RUN_TAG/SIM_PERSIST_ROOT.")
    raise
except ValueError as e:
    print("[precheck] Artifact discovery failed because required files/columns are missing.")
    print("[precheck] Verify merged/shard exports include per_scenario_results and per_eval_trace.")
    raise

print(f"[precheck] Found simulation artifacts at: {run_data.selected_run_prefix}")
print("[precheck] Skipping simulation and running evaluation/analysis only.")

display(run_data.candidates_df)
print("selected_run_prefix:", run_data.selected_run_prefix)

results_df = run_data.results_df.copy()
trace_df = run_data.trace_df.copy()

print("results rows:", len(results_df))
print("trace rows:", len(trace_df))
display(results_df.head())
if len(trace_df) > 0:
    display(trace_df.head())


In [ ]:
# Expanded definition grid
base_risk_defs = standard_risk_definitions()
base_blind_defs = standard_blindspot_definitions()

risk_defs = []
for rd in base_risk_defs:
    if rd.kind == "severity_quantile":
        for q in RISK_SEVERITY_QUANTILES:
            risk_defs.append(replace(rd, name=f"{rd.name}_q{int(round(q*100))}", severity_quantile=float(q)))
    elif rd.kind == "hard_plus_near_miss":
        for t in NEAR_MISS_THRESHOLDS:
            risk_defs.append(replace(rd, name=f"{rd.name}_ttc{str(t).replace('.', 'p')}", near_miss_threshold=float(t)))
    else:
        risk_defs.append(rd)

blind_defs = []
for bd in base_blind_defs:
    if bd.kind == "rare_high_risk_template":
        for q in BLINDSPOT_RARE_QUANTILES:
            blind_defs.append(replace(bd, name=f"{bd.name}_rq{int(round(q*100))}", rare_quantile=float(q)))
    elif bd.kind == "counterfactual_stable_high_risk":
        for r in STABILITY_MIN_RATES:
            blind_defs.append(replace(bd, name=f"{bd.name}_sr{int(round(r*100))}", stability_min_rate=float(r)))
    else:
        blind_defs.append(bd)

metrics_df, labeled_map = evaluate_discovery_grid(
    results_df=results_df,
    trace_df=trace_df,
    risk_definitions=risk_defs,
    blindspot_definitions=blind_defs,
    compute_col=COMPUTE_COL,
    k_values=K_VALUES,
)

method_scores_df = method_score_table(metrics_df)
best_by_definition_df = best_method_per_definition(metrics_df)
hypothesis_df, hypothesis_artifacts = evaluate_discovery_hypotheses(metrics_df, config=HYPOTHESIS_CONFIG)

print("=== Hypothesis Verdicts ===")
display(hypothesis_df)
print("=== Method Scores ===")
display(method_scores_df)


In [ ]:
# Repo-inspired analyses per definition.
repo_rows = []
for definition_key, labeled_df in labeled_map.items():
    base_def = metrics_df[metrics_df["definition_key"] == definition_key][["method", "discovery_efficiency"]].copy()

    plausible_df, natural_mask, natural_meta = plausibility_filtered_summary(
        labeled_df,
        naturalness_config=NATURALNESS_CONFIG,
    )
    coverage_df = cluster_coverage_diversity(labeled_df)
    realism_df = realism_gap_summary(
        labeled_df,
        baseline_method="random",
        compare_on="blindspots",
        feature_cols=("risk_sks", "min_ttc", "delta_l2"),
    )

    combined_def = combine_repo_inspired_method_table(
        base_method_scores_df=base_def,
        plausible_df=plausible_df,
        coverage_df=coverage_df,
        realism_df=realism_df,
    )
    combined_def["definition_key"] = definition_key
    combined_def["naturalness_meta"] = json.dumps(natural_meta)

    rank_def = rulebook_lexicographic_ranking(
        combined_def,
        priorities=RULEBOOK_PRIORITIES,
        maximize=RULEBOOK_MAXIMIZE,
    )
    rank_def["definition_key"] = definition_key
    rank_def["naturalness_meta"] = json.dumps(natural_meta)

    repo_rows.append((combined_def, rank_def))

repo_combined_df = pd.concat([x[0] for x in repo_rows], ignore_index=True) if repo_rows else pd.DataFrame()
repo_rank_df = pd.concat([x[1] for x in repo_rows], ignore_index=True) if repo_rows else pd.DataFrame()

repo_method_summary_df = (
    repo_combined_df.groupby("method", as_index=False)
    .agg(
        mean_discovery_efficiency=("discovery_efficiency", "mean"),
        mean_plausible_efficiency=("plausible_efficiency", "mean"),
        mean_natural_pass_rate=("natural_pass_rate", "mean"),
        mean_coverage_ratio=("coverage_ratio", "mean"),
        mean_entropy=("normalized_entropy", "mean"),
        mean_realism_gap_wd=("wd_mean", "mean"),
    )
    .sort_values("mean_discovery_efficiency", ascending=False)
    .reset_index(drop=True)
    if not repo_combined_df.empty
    else pd.DataFrame()
)

repo_rulebook_summary_df = (
    repo_rank_df.groupby("method", as_index=False)
    .agg(
        mean_rulebook_rank=("rulebook_rank", "mean"),
        std_rulebook_rank=("rulebook_rank", "std"),
        win_fraction=("rulebook_rank", lambda x: float((x == 1).mean())),
    )
    .sort_values("mean_rulebook_rank", ascending=True)
    .reset_index(drop=True)
    if not repo_rank_df.empty
    else pd.DataFrame()
)

print("=== Repo-inspired Method Summary ===")
display(repo_method_summary_df)

print("=== Rulebook Rank Summary ===")
display(repo_rulebook_summary_df)


In [ ]:
# Plots
fig_heat = plot_definition_heatmap(metrics_df, score_col=HYPOTHESIS_CONFIG.score_col, figsize=(11, 6))
if fig_heat is not None:
    plt.show()

fig_rank = plot_rank_heatmap(metrics_df, score_col=HYPOTHESIS_CONFIG.score_col, figsize=(11, 5))
if fig_rank is not None:
    plt.show()

fig_box = plot_method_score_distribution(metrics_df, score_col=HYPOTHESIS_CONFIG.score_col, figsize=(8, 4))
if fig_box is not None:
    plt.show()

for k in K_VALUES:
    col = f"compute_to_k_{int(k)}"
    if col in metrics_df.columns:
        fig_k = plot_time_to_k(metrics_df, k=int(k), figsize=(7, 4))
        if fig_k is not None:
            plt.show()

if not repo_method_summary_df.empty:
    plt.figure(figsize=(8, 4))
    plt.scatter(
        repo_method_summary_df["mean_realism_gap_wd"],
        repo_method_summary_df["mean_discovery_efficiency"],
        s=80,
    )
    for _, r in repo_method_summary_df.iterrows():
        plt.text(r["mean_realism_gap_wd"], r["mean_discovery_efficiency"], str(r["method"]))
    plt.xlabel("Realism gap (WD mean, lower better)")
    plt.ylabel("Discovery efficiency (higher better)")
    plt.title("SEAL-inspired Realism vs Discovery Trade-off")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 4))
    plt.bar(repo_method_summary_df["method"], repo_method_summary_df["mean_plausible_efficiency"])
    plt.title("FREA-inspired Plausibility-Filtered Efficiency")
    plt.ylabel("Plausible blindspot units / compute")
    plt.xlabel("Method")
    plt.tight_layout()
    plt.show()


In [ ]:
Path(EXPORT_DIR).mkdir(parents=True, exist_ok=True)
prefix_name = Path(run_data.selected_run_prefix).name
plots_dir = Path(EXPORT_DIR) / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

out = {
    "metrics": f"{EXPORT_DIR}/{prefix_name}_definition_grid_metrics.csv",
    "method_scores": f"{EXPORT_DIR}/{prefix_name}_method_scores.csv",
    "best_by_definition": f"{EXPORT_DIR}/{prefix_name}_best_method_by_definition.csv",
    "hypothesis": f"{EXPORT_DIR}/{prefix_name}_hypothesis_verdicts.csv",
    "delta_table": f"{EXPORT_DIR}/{prefix_name}_definition_deltas.csv",
    "rank_summary": f"{EXPORT_DIR}/{prefix_name}_rank_summary.csv",
    "winner_frequency": f"{EXPORT_DIR}/{prefix_name}_winner_frequency.csv",
    "repo_definition": f"{EXPORT_DIR}/{prefix_name}_repo_inspired_definition_table.csv",
    "repo_method_summary": f"{EXPORT_DIR}/{prefix_name}_repo_inspired_method_summary.csv",
    "repo_rulebook": f"{EXPORT_DIR}/{prefix_name}_repo_inspired_rulebook_ranks.csv",
    "hypothesis_artifacts": f"{EXPORT_DIR}/{prefix_name}_hypothesis_artifacts.json",
}

metrics_df.to_csv(out["metrics"], index=False)
method_scores_df.to_csv(out["method_scores"], index=False)
best_by_definition_df.to_csv(out["best_by_definition"], index=False)
hypothesis_df.to_csv(out["hypothesis"], index=False)
hypothesis_artifacts["delta_table"].to_csv(out["delta_table"], index=False)
hypothesis_artifacts["rank_summary"].to_csv(out["rank_summary"], index=False)
hypothesis_artifacts["winner_frequency"].to_csv(out["winner_frequency"], index=False)
repo_combined_df.to_csv(out["repo_definition"], index=False)
repo_method_summary_df.to_csv(out["repo_method_summary"], index=False)
repo_rulebook_summary_df.to_csv(out["repo_rulebook"], index=False)

json_payload = {
    "bootstrap": hypothesis_artifacts.get("bootstrap", {}),
    "permutation": hypothesis_artifacts.get("permutation", {}),
    "run_prefix": run_data.selected_run_prefix,
    "rulebook_priorities": RULEBOOK_PRIORITIES,
    "naturalness_config": {
        "max_delta_l2_abs": NATURALNESS_CONFIG.max_delta_l2_abs,
        "max_delta_l2_quantile": NATURALNESS_CONFIG.max_delta_l2_quantile,
        "max_fallback_ratio": NATURALNESS_CONFIG.max_fallback_ratio,
        "max_actor_fallback_ratio": NATURALNESS_CONFIG.max_actor_fallback_ratio,
        "require_rollout_feasible": NATURALNESS_CONFIG.require_rollout_feasible,
        "require_no_feasibility_violation": NATURALNESS_CONFIG.require_no_feasibility_violation,
        "min_dist_non_null_ratio": NATURALNESS_CONFIG.min_dist_non_null_ratio,
    },
}
with open(out["hypothesis_artifacts"], "w") as f:
    json.dump(json_payload, f, indent=2)

# Save static plot files.
save_figure(plot_definition_heatmap(metrics_df, score_col=HYPOTHESIS_CONFIG.score_col, figsize=(11, 6)), str(plots_dir / f"{prefix_name}_definition_heatmap.png"))
save_figure(plot_rank_heatmap(metrics_df, score_col=HYPOTHESIS_CONFIG.score_col, figsize=(11, 5)), str(plots_dir / f"{prefix_name}_rank_heatmap.png"))
save_figure(plot_method_score_distribution(metrics_df, score_col=HYPOTHESIS_CONFIG.score_col, figsize=(8, 4)), str(plots_dir / f"{prefix_name}_method_score_boxplot.png"))
for k in K_VALUES:
    col = f"compute_to_k_{int(k)}"
    if col in metrics_df.columns:
        save_figure(plot_time_to_k(metrics_df, k=int(k), figsize=(7, 4)), str(plots_dir / f"{prefix_name}_time_to_k_{int(k)}.png"))

print("Exported tables and plots:")
for k, v in out.items():
    print("-", k, ":", v)
print("- plots_dir:", str(plots_dir))


## Reporting Guidance
- Report hypothesis verdicts first.
- Add repo-inspired summary table (plausibility, diversity, realism-gap, rulebook rank) as core ablation.
- Put full definition-grid in appendix for transparency.
